# IESO Coincident Peak Prediction — Feature Engineering

This notebook transforms raw demand and weather data into ML-ready features.
Feature categories:
- **Weather features:** humidex, cooling degree hours, rolling averages
- **Temporal features:** calendar indicators, Ontario holidays
- **Demand momentum:** lagged demand, rolling windows
- **Peak context:** running threshold tracker, peaks accumulated

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path
from sklearn.feature_selection import mutual_info_regression

warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_theme(style='whitegrid', font_scale=1.1)

# Paths
PROJECT_ROOT = Path(r'C:/wamp64/www/Spec_Driven_Dev_Website')
DATA_DIR = PROJECT_ROOT / 'notebooks' / 'source' / 'data'

print(f'Loading data from: {DATA_DIR}')

In [ ]:
# Load cleaned hourly dataset from Notebook 1
hourly = pd.read_parquet(DATA_DIR / 'ieso_demand_weather_2010_2025.parquet')
peaks = pd.read_parquet(DATA_DIR / 'ieso_peak_labels.parquet')

hourly['Date'] = pd.to_datetime(hourly['Date'])
hourly['datetime'] = pd.to_datetime(hourly['datetime'])
peaks['date'] = pd.to_datetime(peaks['date'])

print(f'Hourly dataset: {len(hourly):,} rows')
print(f'Peak labels: {len(peaks)} records across {peaks["base_period"].nunique()} base periods')
print(f'Date range: {hourly["Date"].min()} to {hourly["Date"].max()}')

## Temporal Features

Calendar indicators including Ontario statutory holidays. Holidays behave
like weekends for demand purposes — commercial and industrial loads drop.

In [ ]:
# Ontario statutory holidays (fixed-date and rule-based)
def get_ontario_holidays(year):
    """Generate Ontario statutory holiday dates for a given year."""
    from datetime import date
    import calendar
    
    holidays = []
    
    # Fixed-date holidays
    holidays.append(date(year, 1, 1))   # New Year's Day
    holidays.append(date(year, 7, 1))   # Canada Day
    holidays.append(date(year, 12, 25)) # Christmas Day
    holidays.append(date(year, 12, 26)) # Boxing Day
    
    # Family Day: 3rd Monday of February
    c = calendar.monthcalendar(year, 2)
    mondays = [week[calendar.MONDAY] for week in c if week[calendar.MONDAY] != 0]
    holidays.append(date(year, 2, mondays[2]))
    
    # Good Friday: 2 days before Easter Sunday
    # Easter calculation (Anonymous Gregorian algorithm)
    a = year % 19
    b = year // 100
    c_val = year % 100
    d = b // 4
    e = b % 4
    f = (b + 8) // 25
    g = (b - f + 1) // 3
    h = (19 * a + b - d - g + 15) % 30
    i = c_val // 4
    k = c_val % 4
    l = (32 + 2 * e + 2 * i - h - k) % 7
    m = (a + 11 * h + 22 * l) // 451
    month = (h + l - 7 * m + 114) // 31
    day = ((h + l - 7 * m + 114) % 31) + 1
    easter = date(year, month, day)
    good_friday = easter - timedelta(days=2)
    holidays.append(good_friday)
    
    # Victoria Day: Monday before May 25
    may25 = date(year, 5, 25)
    days_since_monday = (may25.weekday()) % 7
    victoria_day = may25 - timedelta(days=days_since_monday) if may25.weekday() != 0 else may25
    holidays.append(victoria_day)
    
    # Civic Holiday: 1st Monday of August
    c = calendar.monthcalendar(year, 8)
    mondays = [week[calendar.MONDAY] for week in c if week[calendar.MONDAY] != 0]
    holidays.append(date(year, 8, mondays[0]))
    
    # Labour Day: 1st Monday of September
    c = calendar.monthcalendar(year, 9)
    mondays = [week[calendar.MONDAY] for week in c if week[calendar.MONDAY] != 0]
    holidays.append(date(year, 9, mondays[0]))
    
    # Thanksgiving: 2nd Monday of October
    c = calendar.monthcalendar(year, 10)
    mondays = [week[calendar.MONDAY] for week in c if week[calendar.MONDAY] != 0]
    holidays.append(date(year, 10, mondays[1]))
    
    return [pd.Timestamp(h) for h in holidays]

from datetime import timedelta

# Build holiday set for all years
all_holidays = set()
for year in range(2010, 2027):
    all_holidays.update(get_ontario_holidays(year))

# Add temporal features
hourly['month'] = hourly['Date'].dt.month
hourly['day_of_week'] = hourly['Date'].dt.dayofweek  # 0=Monday, 6=Sunday
hourly['day_of_year'] = hourly['Date'].dt.dayofyear
hourly['week_of_year'] = hourly['Date'].dt.isocalendar().week.astype(int)
hourly['is_weekday'] = (hourly['day_of_week'] < 5).astype(int)
hourly['is_ontario_holiday'] = hourly['Date'].isin(all_holidays).astype(int)
# Effective business day: weekday AND not a holiday
hourly['is_business_day'] = ((hourly['is_weekday'] == 1) & 
                              (hourly['is_ontario_holiday'] == 0)).astype(int)

print(f'Temporal features added.')
print(f'Ontario holidays identified: {hourly["is_ontario_holiday"].sum()} hourly records')
print(f'Business days: {hourly["is_business_day"].sum():,} hours '
      f'({hourly["is_business_day"].mean()*100:.1f}%)')

## Weather Features

Transform raw temperature and humidity into domain-relevant indicators:
- **Humidex:** perceived temperature accounting for humidity
- **Cooling Degree Hours (CDH):** cumulative heat exposure above 18°C
- **Rolling averages:** multi-day thermal mass effects

In [ ]:
def compute_humidex(temp_c, dewpoint_c):
    """Calculate humidex from temperature and dewpoint.
    
    H = T + (5/9) * (6.11 * exp(5417.7530 * (1/273.16 - 1/(273.15+Td))) - 10)
    """
    e = 6.11 * np.exp(5417.7530 * (1.0/273.16 - 1.0/(273.15 + dewpoint_c)))
    h = temp_c + (5.0/9.0) * (e - 10.0)
    # Humidex is only meaningful when T > 20°C
    return np.where(temp_c >= 20, h, temp_c)

# Hourly weather features
hourly['humidex'] = compute_humidex(
    hourly['temperature_c'].values, 
    hourly['dewpoint_c'].values
)

# Cooling Degree Hours: max(0, T - 18) per hour
hourly['cdh'] = np.maximum(0, hourly['temperature_c'] - 18.0)

# Daily aggregations
daily_weather = hourly.groupby('Date').agg({
    'temperature_c': ['max', 'mean', 'min'],
    'humidex': 'max',
    'dewpoint_c': 'mean',
    'relative_humidity': 'mean',
    'cdh': 'sum',  # daily CDH = sum of hourly CDH
    'shortwave_radiation': 'sum',  # daily solar insolation
}).reset_index()
daily_weather.columns = ['Date', 'daily_max_temp', 'daily_mean_temp', 'daily_min_temp',
                          'daily_max_humidex', 'daily_mean_dewpoint',
                          'daily_mean_rh', 'daily_cdh', 'daily_solar']

# Rolling averages (thermal mass effects)
daily_weather = daily_weather.sort_values('Date').reset_index(drop=True)
daily_weather['temp_3day_avg'] = daily_weather['daily_max_temp'].rolling(3, min_periods=1).mean()
daily_weather['temp_5day_avg'] = daily_weather['daily_max_temp'].rolling(5, min_periods=1).mean()
daily_weather['cdh_3day_avg'] = daily_weather['daily_cdh'].rolling(3, min_periods=1).mean()

print(f'Daily weather features: {len(daily_weather)} days')
print(f'\nFeature statistics (summer months only):')
summer_mask = daily_weather['Date'].dt.month.isin([6, 7, 8])
print(daily_weather.loc[summer_mask, ['daily_max_temp', 'daily_max_humidex', 'daily_cdh']].describe().round(1))

## Demand Momentum Features

Lagged and rolling demand features capture system-level momentum.
These are available by the morning of the prediction day.

In [ ]:
# Daily demand summary
daily_demand = hourly.groupby('Date').agg({
    'Ontario Demand': ['max', 'mean'],
    'base_period': 'first',
    'is_business_day': 'first',
    'is_weekday': 'first',
    'is_ontario_holiday': 'first',
    'month': 'first',
    'day_of_week': 'first',
    'day_of_year': 'first',
}).reset_index()
daily_demand.columns = ['Date', 'daily_max_demand', 'daily_mean_demand',
                         'base_period', 'is_business_day', 'is_weekday',
                         'is_ontario_holiday', 'month', 'day_of_week', 'day_of_year']

# Morning demand ramp: demand at Hour 10 (10 AM) as early predictor
morning_demand = hourly[hourly['Hour'] == 10][['Date', 'Ontario Demand']].copy()
morning_demand.columns = ['Date', 'demand_at_10am']
daily_demand = daily_demand.merge(morning_demand, on='Date', how='left')

# Demand at Hour 13 (1 PM) for intraday update
afternoon_demand = hourly[hourly['Hour'] == 13][['Date', 'Ontario Demand']].copy()
afternoon_demand.columns = ['Date', 'demand_at_1pm']
daily_demand = daily_demand.merge(afternoon_demand, on='Date', how='left')

# Lagged features (shifted by 1 day so they're available in the morning)
daily_demand = daily_demand.sort_values('Date').reset_index(drop=True)
daily_demand['prev_day_max_demand'] = daily_demand['daily_max_demand'].shift(1)
daily_demand['prev_day_mean_demand'] = daily_demand['daily_mean_demand'].shift(1)

# Rolling windows
daily_demand['rolling_7d_max_demand'] = (
    daily_demand['daily_max_demand'].shift(1).rolling(7, min_periods=1).max()
)
daily_demand['rolling_7d_mean_demand'] = (
    daily_demand['daily_max_demand'].shift(1).rolling(7, min_periods=1).mean()
)

# Year-over-year growth (same month, previous year average)
monthly_avg = daily_demand.groupby([daily_demand['Date'].dt.year, 'month'])['daily_max_demand'].mean()
monthly_avg = monthly_avg.reset_index()
monthly_avg.columns = ['year', 'month', 'monthly_avg_demand']

print(f'Daily demand features: {len(daily_demand)} days')
print(f'\nLagged feature sample:')
print(daily_demand[['Date', 'daily_max_demand', 'prev_day_max_demand', 
                     'rolling_7d_max_demand']].dropna().tail(5).to_string(index=False))

## Peak Context Features

These features are unique to the coincident peak prediction problem. They track
the running top-5 peak state within each base period, providing the displacement
threshold that defines whether a day could produce a new peak.

**Critical:** these are computed sequentially within each base period to prevent
look-ahead bias.

In [ ]:
# Build peak context features within each base period
# Using ground truth peak labels from the archive
top5 = peaks[peaks['rank'] <= 5].copy()

def compute_peak_context(daily_df, peaks_df):
    """Compute running peak tracker features for each base period.
    
    For each day, computes:
    - current_5th_peak: the 5th-highest demand seen so far in this base period
    - peaks_so_far: how many of the eventual top-5 peaks have occurred
    - days_since_last_peak: days since the last top-5 peak in this base period
    - max_demand_so_far: highest daily max demand in this base period to date
    """
    results = []
    
    for bp in sorted(daily_df['base_period'].unique()):
        bp_mask = daily_df['base_period'] == bp
        bp_days = daily_df[bp_mask].sort_values('Date').copy()
        
        # Get actual peak dates for this base period
        bp_peaks = peaks_df[peaks_df['base_period'] == bp]
        peak_dates = set(bp_peaks['date'].dt.date) if len(bp_peaks) > 0 else set()
        
        running_top5 = []  # Track top-5 demands seen so far
        last_peak_date = None
        
        for idx, row in bp_days.iterrows():
            current_date = row['Date'].date() if hasattr(row['Date'], 'date') else row['Date']
            
            # Current threshold (5th highest demand seen so far)
            if len(running_top5) >= 5:
                current_5th = sorted(running_top5, reverse=True)[4]
            elif len(running_top5) > 0:
                current_5th = min(running_top5)
            else:
                current_5th = 0
            
            # Days since last peak
            if last_peak_date is not None:
                days_since = (pd.Timestamp(current_date) - last_peak_date).days
            else:
                days_since = -1  # No peak yet this base period
            
            # Max demand so far
            max_so_far = max(running_top5) if running_top5 else 0
            
            results.append({
                'Date': row['Date'],
                'current_5th_peak': current_5th,
                'peaks_so_far': sum(1 for d in running_top5 
                                   if d >= (sorted(running_top5, reverse=True)[4] 
                                           if len(running_top5) >= 5 else 0)),
                'num_top5_seen': min(len(running_top5), 5),
                'days_since_last_peak': days_since,
                'max_demand_so_far': max_so_far,
            })
            
            # Update running tracker AFTER computing features for today
            # (prevents look-ahead bias)
            demand_today = row['daily_max_demand']
            if not np.isnan(demand_today):
                running_top5.append(demand_today)
                running_top5 = sorted(running_top5, reverse=True)[:10]  # Keep top 10
            
            # Track if today was a peak day
            if current_date in peak_dates:
                last_peak_date = pd.Timestamp(current_date)
    
    return pd.DataFrame(results)

peak_context = compute_peak_context(daily_demand, top5)
print(f'Peak context features computed: {len(peak_context)} days')
print(f'\nSample (summer 2024):')
sample_mask = (peak_context['Date'] >= '2024-06-01') & (peak_context['Date'] <= '2024-08-31')
print(peak_context[sample_mask].head(10).to_string(index=False))

## Target Variable Construction

Create the regression target (daily max demand) and binary classification labels
(is this a top-5 peak day?) from ground truth data.

In [ ]:
# Mark peak days in the daily dataset
peak_dates_set = set(top5['date'].dt.date)
daily_demand['is_peak_day'] = daily_demand['Date'].apply(
    lambda d: 1 if d.date() in peak_dates_set else 0
)

print(f'Peak day labels:')
print(f'  Total peak days: {daily_demand["is_peak_day"].sum()}')
print(f'  Total non-peak days: {(daily_demand["is_peak_day"] == 0).sum()}')
print(f'  Peak day rate: {daily_demand["is_peak_day"].mean()*100:.2f}%')
print(f'\nPeak days per base period:')
print(daily_demand.groupby('base_period')['is_peak_day'].sum().to_string())

## Assemble Feature Matrix

Merge all feature groups into a single daily-resolution feature matrix.

In [ ]:
# Merge all features
features = daily_demand.merge(daily_weather, on='Date', how='left')
features = features.merge(peak_context, on='Date', how='left')

# Drop rows with insufficient data (first few days of each base period)
features = features.dropna(subset=['prev_day_max_demand', 'daily_max_temp']).copy()

print(f'Feature matrix: {len(features)} days x {len(features.columns)} columns')
print(f'\nFeature columns:')
for col in sorted(features.columns):
    print(f'  {col}: {features[col].dtype}')

print(f'\nDate range: {features["Date"].min()} to {features["Date"].max()}')
print(f'Base periods: {sorted(features["base_period"].unique())}')

## Feature Correlation Matrix

In [ ]:
# Select numeric features for correlation
feature_cols = [
    'daily_max_temp', 'daily_mean_temp', 'daily_max_humidex', 'daily_cdh',
    'daily_mean_rh', 'daily_mean_dewpoint', 'temp_3day_avg', 'temp_5day_avg',
    'cdh_3day_avg', 'daily_solar',
    'month', 'day_of_week', 'is_business_day', 'day_of_year',
    'prev_day_max_demand', 'rolling_7d_max_demand', 'rolling_7d_mean_demand',
    'demand_at_10am',
    'current_5th_peak', 'max_demand_so_far', 'days_since_last_peak',
    'daily_max_demand'
]

corr_data = features[feature_cols].dropna()
corr_matrix = corr_data.corr()

fig, ax = plt.subplots(figsize=(16, 14))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True, ax=ax,
            annot_kws={'fontsize': 7})
ax.set_title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.savefig(DATA_DIR / 'feature_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Highlight high correlations with target
target_corr = corr_matrix['daily_max_demand'].drop('daily_max_demand').sort_values(ascending=False)
print('\nCorrelation with daily_max_demand:')
print(target_corr.to_string())

## Feature Importance (Mutual Information)

In [ ]:
# Mutual information scores
predictor_cols = [c for c in feature_cols if c != 'daily_max_demand']
mi_data = features[predictor_cols + ['daily_max_demand']].dropna()

X_mi = mi_data[predictor_cols].values
y_mi = mi_data['daily_max_demand'].values

mi_scores = mutual_info_regression(X_mi, y_mi, random_state=42)
mi_df = pd.DataFrame({'Feature': predictor_cols, 'MI Score': mi_scores})
mi_df = mi_df.sort_values('MI Score', ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#d32f2f' if 'temp' in f or 'humidex' in f or 'cdh' in f or 'solar' in f
          else '#1565C0' if 'demand' in f or 'peak' in f
          else '#4CAF50' for f in mi_df['Feature']]
ax.barh(range(len(mi_df)), mi_df['MI Score'], color=colors)
ax.set_yticks(range(len(mi_df)))
ax.set_yticklabels(mi_df['Feature'])
ax.set_xlabel('Mutual Information Score')
ax.set_title('Feature Importance for Daily Max Demand Prediction\n'
             '(Red=Weather, Blue=Demand, Green=Calendar)')

plt.tight_layout()
plt.savefig(DATA_DIR / 'feature_importance_mi.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 10 features by mutual information:')
print(mi_df.sort_values('MI Score', ascending=False).head(10).to_string(index=False))

## Save Feature-Engineered Dataset

In [ ]:
# Save the complete feature matrix
features.to_parquet(DATA_DIR / 'ieso_features_daily.parquet', index=False)
print(f'Saved feature-engineered dataset: {len(features)} days x {len(features.columns)} columns')
print(f'  -> {DATA_DIR / "ieso_features_daily.parquet"}')

# Also save the hourly dataset with weather features for Notebook 5
hourly_with_features = hourly.copy()
hourly_with_features.to_parquet(DATA_DIR / 'ieso_hourly_with_features.parquet', index=False)
print(f'\nSaved hourly dataset with features: {len(hourly_with_features):,} rows')
print(f'  -> {DATA_DIR / "ieso_hourly_with_features.parquet"}')

print('\n=== Notebook 2 complete ===')